# Quantum Circuit Compilation from Source

This notebook assembles circuits for the three benchmarks of the competition and checks each one with the official verifier at its full number of shots. The benchmarks are an inverse-square-root phase for Coulomb interactions in fusion plasmas, a Reed–Solomon decoder for Decoded Quantum Interferometry (DQI), and the angle lookups of a FeMoCo block encoding. All circuits are kickmix circuits — reversible classical gates, phase gates and measurement-based uncomputation — so a classical simulator can check them input by input.

The score is the geometric mean of $\text{cost}_\text{ref}/\text{cost}$ over the three tasks. The costs are defined by the organizers as follows.

| Task | Cost | Reference circuit | This notebook |
|---|---|---|---|
| Inverse square root | Q × (Toffoli + T / 4) | Q 260, 7,735 Toffolis | Q 63, 1,625 Toffoli-equivalents |
| DQI decoder | R(ε₊) × Q² × T | Q 1,893, 5.73 M Toffolis | Q 889, 0.44 M Toffolis |
| FeMoCo lookups | λ_eff × T_total × Q_total | Q 730, 93,237 Toffolis, λ_eff 49.30 | Q 430, 16,274 Toffolis, λ_eff 48.57 |

Q is the qubit count. In the first row T counts T gates, and synthesized rotations are priced in T gates. In the second row T is the mean Toffoli count per shot, and R(ε₊) grows with the decoder's measured failure rate. T_total and Q_total cover the whole block encoding, including the organizers' state-preparation costs. Because the score is a geometric mean, a 1% saving in any task raises it by about 0.33%; what differs between tasks is how much circuit a percent costs. Near 880 qubits one DQI qubit is worth about 0.23% of its cost, while one inverse-square-root qubit near 63 is worth about 1.6%.

Each section below follows the same order: why the subroutine matters, what the circuit computes, where the savings come from, and which approximation is used and how much of the accuracy budget it spends.

In [ ]:
import json, math, os, pathlib, sys
os.environ.update(OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')
import pandas as pd
from IPython.display import display

INPUT = pathlib.Path(os.environ.get('QCC_INPUT', '/kaggle/input'))
WORK = pathlib.Path(os.environ.get('QCC_WORK', '/kaggle/working'))
SUB = pathlib.Path(os.environ.get('QCC_SUB', '/tmp/qcc_submission'))
SLUG = 'google-verifiable-quantum-circuit-compilation'
COMP = next(p for p in [INPUT / 'competitions' / SLUG, INPUT / SLUG] if (p / 'kickmix').exists())
# The chemistry numeric input is located by the build cell.
WORKERS = int(os.environ.get('QCC_WORKERS', os.cpu_count() or 4))
for d in (SUB, WORK / 'src' / 'gen_invsqrt_r2', WORK / 'src' / 'gen_dqi_v6' / 'ref'):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
REFERENCE = {'invsqrt': 2076774.233715282, 'dqi': 24439474177519.598, 'dfthcsos': 3355599588.3155365}

for _d in []:
    (WORK / _d).mkdir(parents=True, exist_ok=True)

In [ ]:
import hashlib, lzma
OWN = next(p.parent for p in INPUT.rglob('femoco_thc_r5.npz'))
OPEN = OWN
FILES = {
    'dfthcsos/angle_k_0.kmb': ('open_circuits_v38/dfthcsos__angle_k_0.kmb', '6f516cba96b50d3e57b4c07574b1e4026f4392b8b904b6f82586d9a18b9f7a14'),
    'dfthcsos/angle_k_1.kmb': ('open_circuits_v38/dfthcsos__angle_k_1.kmb', 'd10095ee50669b354499dc5674002dd9d6796b3e07d43cefec24d00530a5dde2'),
    'dfthcsos/angle_k_10.kmb': ('open_circuits_v38/dfthcsos__angle_k_10.kmb', 'a5f62a17d6387e9f5c3b30e7b9dc5a6f6ab4d6196192452a5f6d7fc47bc8d81b'),
    'dfthcsos/angle_k_11.kmb': ('open_circuits_v38/dfthcsos__angle_k_11.kmb', '2cf26d3530623a33c84d241f6ae00449c4d902094b7de6188ea50d76f7640af4'),
    'dfthcsos/angle_k_12.kmb': ('open_circuits_v38/dfthcsos__angle_k_12.kmb', '9a1b34be8b56a08c6b1a3e1ad6ce4f6108229dddf48256e249585aeecab175b1'),
    'dfthcsos/angle_k_13.kmb': ('open_circuits_v38/dfthcsos__angle_k_13.kmb', 'da942c0f2bae0f12e62816e52998ea14eaeb010fcccd26c987b76abfa8049ff0'),
    'dfthcsos/angle_k_14.kmb': ('open_circuits_v38/dfthcsos__angle_k_14.kmb', 'b7b58efed7192e2ff1e297c3647429bc90476b352e0069fe0b737563a5a7713f'),
    'dfthcsos/angle_k_2.kmb': ('open_circuits_v38/dfthcsos__angle_k_2.kmb', '2e8dfdeb69c496e9e510fe99722153663760980d45c35ad9b72e79ba799dde49'),
    'dfthcsos/angle_k_3.kmb': ('open_circuits_v38/dfthcsos__angle_k_3.kmb', '4202b4526109487d17b0b3a0a320007fded51c05e25e54f3d7f7cd98592b1ded'),
    'dfthcsos/angle_k_4.kmb': ('open_circuits_v38/dfthcsos__angle_k_4.kmb', 'a2b685bcae6a1b0457c97389693935c44e0872f41add1abb4ab84caccfc754ab'),
    'dfthcsos/angle_k_5.kmb': ('open_circuits_v38/dfthcsos__angle_k_5.kmb', '174f8ba8efda5aed6f3ec0fea5907e64491306e884b6e382ccc2e37161b1bc68'),
    'dfthcsos/angle_k_6.kmb': ('open_circuits_v38/dfthcsos__angle_k_6.kmb', 'eeff5947bcfca7e13aeb682ad912b8207407e4591ecc47041ce42854418eb071'),
    'dfthcsos/angle_k_7.kmb': ('open_circuits_v38/dfthcsos__angle_k_7.kmb', 'b92d54c4cb681f0b7b78db7503d52ba883a5cf77fa7647005157973b2b0be4f4'),
    'dfthcsos/angle_k_8.kmb': ('open_circuits_v38/dfthcsos__angle_k_8.kmb', '6e6aaa69b8a4785efbd2802f4f0490da30fba5ab73224c6fca08f5c398e07fd5'),
    'dfthcsos/angle_k_9.kmb': ('open_circuits_v38/dfthcsos__angle_k_9.kmb', '23b4022af8e417c64b7646f5e9360fdaf2ddc876e78abfd60d847217670d6d1b'),
    'dfthcsos/submission.npz': ('open_circuits_v39/dfthcsos__submission.npz', '18f3c5a996afdfee50fa41ab2d95fa42d14bf9f9194482ef0a895243d23f3c53'),
    'dqi.kmb': ('open_circuits_v39/dqi.kmb.xz', '6349d2bc82eb1ef3a4e8c52c71a312823d756532517463b5e18eb25cc81195a8'),
    'invsqrt.kmb': ('open_circuits_v31/invsqrt.kmb', '9a8627c812833a6feca6eb82239f38d1a9a5e1eee299a9a7af9abd85e36339be'),
}
for name, (stored, pin) in FILES.items():
    data = (OPEN / stored).read_bytes()
    if stored.endswith('.xz'):
        data = lzma.decompress(data)
    assert hashlib.sha256(data).hexdigest() == pin, name
    (SUB / name).parent.mkdir(parents=True, exist_ok=True)
    (SUB / name).write_bytes(data)
sys.path.insert(0, str(WORK / 'src'))
import shutil
shutil.copytree(next(OWN.rglob('qcc_pipeline.py')).parent, WORK / 'src', dirs_exist_ok=True)
import qcc_pipeline as P
P.setup(COMP)
RESULTS = P.verify_all(SUB, 'notebook-verification', workers=WORKERS)
assert all(r['status'] == 'PASS' for r in RESULTS.values())
(WORK / 'verification.json').write_text(json.dumps(RESULTS, indent=1, default=str))
P.write_zip(SUB, WORK / 'submission.zip')

## 1. Inverse square root

**Role.** Stopping-power simulations of warm dense plasmas evaluate a pairwise Coulomb potential. The quantum subroutine applies the phase $e^{2\pi i/\sqrt{x}}$ to a 22-bit input $x$, and every gate in it is repeated for every particle pair and time step.

**Computation.** Binary shift stages normalize $x$ to a mantissa $M \in [1, 2)$. Their flags give the shift, with $e$ its parity and $s$ half of it:

$$\frac{1}{\sqrt{x}} = 2^{\,s-10}\, y, \qquad y = \frac{2^{(e-1)/2}}{\sqrt{M}} \in (0.5, 1].$$

A lookup addressed by $e$ and the high mantissa bits loads the coefficients of a centered quadratic, and the low bits form an offset whose digits are all $\pm 1$:

$$\delta = \sum_i (2d_i - 1)\, 2^{\,i-22}, \qquad R = K_1 - C_2\,\delta, \qquad y = K_0 - R\,\delta .$$

**Savings.**
- Because every digit is $\pm 1$, each Horner row is a plain addition or subtraction rather than a controlled multiplication.
- The derivative register is split into a short residual and coarse bits, the constant high part of the result is phased early, the curvature is looked up per group of segments and erased by measurement, and the lowest accumulator bit is kept virtual.
- The output phase is fused: shifts by one and two positions are applied with controlled swaps, and each of the 44 calibrated rotations reads its bit through a single temporary.
- The main coefficient lookup is synthesized with 172 AND gates instead of 186, and the six product blocks of one later window with 25 instead of 30, with the same output tables; together they lower the cost by about 1.5%.\n- Most of the width reduction comes from storage and uncomputation alone, with the polynomial and all rotation angles unchanged. Lookup outputs that are Boolean functions of other outputs are recomputed where they are used, carry prefixes are finished and erased early, increment-tail workspaces are removed by temporarily erasing the top accumulator bit, and input bits and normalization flags are parked in freed slots. Each identity is checked on all 3,133,452 legal inputs. The circuit has 63 qubits and executes about 1,330 Toffolis and 41 fine rotations per input.

**Approximation.** The quadratic coefficients use short precisions fitted on the reachable inputs, and the rotation angles are calibrated over the whole domain. The worst phase error stays just below the tolerance after rotation synthesis, so almost the whole budget is spent. The verifier tests every legal input 64 times.

## 2. DQI decoder

**Role.** Decoded Quantum Interferometry turns an optimization problem over a finite field into the decoding of a Reed–Solomon code. Its dominant cost is a reversible decoder that runs in superposition.

**Computation.** Seventy 12-bit registers hold the syndromes $S_i = \sum_j e_j \alpha^{(i+1)j}$ of an unknown error vector $e$ over GF($2^{12}$) with at most 35 nonzero entries among 4,095 positions. Beside them sit 4,095 classical words $M_j$. The circuit multiplies the phase by $(-1)^{\sum_j |e_j \wedge M_j|}$, returns the syndromes and leaves every other qubit clean: it decodes $e$ in superposition, applies the phase and undoes the decoding. The locator $\sigma$ and evaluator $\Omega$ solve

$$S(z)\,\sigma(z) \equiv \Omega(z) \pmod{z^{70}}.$$

**Savings.**
- The syndrome words are transformed in place, so the decoder needs only 49 qubits beyond its 840 input qubits.
- **Confluent Hermite extraction.** Instead of reading the syndrome generating function $G$ at 35 distinct nodes, the decoder reads $G$ at one node and $G$ together with its derivative at 17 double nodes. This carries the same information, as in Hermite interpolation, but each double node yields two continued-fraction heads, so the extraction stage needs nearly half as many steps at a somewhat higher price per step. A later jet is updated by one jet inversion, $J(u, v) = (u^{-1}, v\,u^{-2})$ for $u \ne 0$ (in characteristic 2 the value and derivative of $1/G$), plus fixed linear maps. The 289 jet inversions carry about 39% of all Toffolis.
- **A cheaper jet inversion.** The inverse of $u \in \mathrm{GF}(2^{24})$ is taken through its norm $N = u\bar u \in \mathrm{GF}(2^{12})$. Forty-two Toffolis compute $N$ together with quadratic corrections in a 21-qubit cache; the scalar inverse of $N$ descends a tower of subfields to $\mathrm{GF}(64)$, where an inverse with its zero flag costs 18 Toffolis; and the cached corrections cut the two final products from 84 to 70 Toffolis. A forward jet costs 269 Toffolis and its reverse 389, which also pays the phase of the measured inverse with the zero flag it already holds. The jet's workspace, 43 qubits on top of the syndromes, leaves room for further trades between width and Toffolis elsewhere in the decoder.
- **Narrow folds.** The continued-fraction heads are folded into the error locator with zero-power padding of short locators and retained norms in each division.
- **Streamed conversion.** The conversion that turns the locator and evaluator into error values is routed differently. From degree 7 up it pivots only the last two words of one polynomial, uses shifted products that are never formed explicitly, and swaps the two polynomial banks in place instead of rotating them into canonical form; at degrees 2 to 6 it restores the measured heads of the full products. A full head would need one more word than the bank holds, so it is evaluated with an unknown alias of the high coefficient, the low head is installed first, and the alias is removed linearly. Together with smaller truncated products (99 Toffolis for a 2×2 product kept to 2 terms) and the erasure of the first nonlinear product of each replay chain while both operands are still available, this saves about 9,800 Toffolis per shot at the same width. Finally, eight qubits are given back in four places, each trading Toffolis for width: a paired chart for the high components in most extraction chains, an inverse computed in reverse in the later groups, a synthesis division that consumes its norm and rebuilds it later instead of keeping another register bank, and a retimed schedule for the measured heads. Every stage then peaks at 889 qubits or fewer; the shot costs about 7,100 Toffolis more, and $Q^2 T$ falls by 0.15%.
- The locator and evaluator then give the error values through $A \leftarrow A / B' \bmod B$, a scan applies each position's phase where the locator vanishes, and the whole computation is undone. A shot uses about 437,900 Toffolis on average, less than a tenth of the reference decoder.

**Approximation.** Rare degenerate pivot patterns are not handled exactly. Under the competition distribution they cause a few phase failures in 102,400 shots, which the verifier prices through $R(\varepsilon_+)$, a Clopper–Pearson bound at confidence $1 - 2^{-128}$; with no failures $R = 1.191$.

## 3. FeMoCo angle lookups

**Role.** FeMoCo is the active site of nitrogenase. Its block encoding is the core of a quantum chemistry algorithm for nitrogen fixation. The lookup circuits stream the 75 Givens-rotation angles of every factor vector into word registers.

**Factorization.** The two-body tensor is written with eight groups of 143 unit vectors and rank 60, $(R, B, C) = (8, 143, 60)$, which gives 76 + 8 × 143 = 1,220 lookup addresses, and every angle is stored with **8 bits**. The short words are the main lever: the organizers' state-preparation cost model sizes its temporary workspace by the word width, so the whole block encoding needs 430 qubits instead of about 740. The groups come from a larger design with 154 vectors per group by exact compensated deletion: each removed vector is followed by a refit of the remaining ones that moves the represented Hamiltonian as little as possible. With 143 vectors a group fills nine 16-way blocks of the inner state preparation instead of ten, which saves about 500 Toffolis per step, and rank 60 saves 150 more. The fit aims at a slightly biased target, with a small positive coherent term that keeps $E_\text{gap}$ above its floor, while both errors are always measured against the true Hamiltonian. With every word fixed, the weights and identity offsets are then fitted for the lowest $\lambda_\text{eff}$ within the error and gap limits, and two further rounds requantize the one-body words in the fitted basis and refit the weights. That fit leaves about 4 mHa of the one-body error unused; a last refit of the weights and offsets at fixed words spends it on the normalization. Finally, for each group the identity offsets are appended to the weights, $A_r = [W_r; H_{2,I,r}]$, and the columns are rotated by an orthogonal matrix, $A_r' = A_r Q_r$. This leaves $A_r A_r^\top$, and therefore the two-body tensor, the absorbed one-body term and the SOS energy, exactly unchanged, but it changes the normalization term $\tfrac14 \sum_{r,c} \lVert A_r[:,c] \rVert_1^2$; exact pair rotations, then rotations of several columns at once in the Cayley parametrization and a fine continuation, lower the normalization further without touching any angle word or circuit. Together the refit and the gauges take $\lambda_\text{eff}$ from 48.764 to 48.604. A last stage, started from Ahmed Berat Özer's public weights, optimizes $\log\lambda_\text{eff}$ over all active coefficients of $[W; H_{2,I}]$ independently, with augmented-Lagrangian multipliers for both errors and the gap, and reaches 48.573.

**Circuits.** Circuit $k$ XORs into five 8-bit word registers the XOR of the words of batches $k$ and $k-1$, as a function of the 11-bit address. Because the set of legal addresses is closed under bit subsets, every output bit is an XOR of address monomials, and one Toffoli computes the AND of two XOR combinations, so it covers two products at once through $(h_1+l_2)(h_2+l_1) = h_1l_1 + h_2l_2 + h_1h_2 + l_1l_2$ over GF(2). Two ideas lower the count:
- **Coded low bits.** In 11 of the 15 tables, the lowest bit of each word is, on every block of 16 consecutive addresses, an affine function of the address bits (a first-order Reed–Muller code word), and in five of them the next bit is coded the same way. Such bits cost almost nothing to load. The words were quantized to 8 bits jointly with these codes, so the accuracy is paid once, in the factorization.
- **XOR targets.** Each product's Toffoli targets one of the word bits it changes, between two fans of CNOTs to the others, so the product is added to all of them. In the frozen design no word qubit serves as workspace, which makes every circuit the oracle $|a\rangle|z\rangle \to |a\rangle|z \oplus \delta_k(a)\rangle$ for arbitrary words $z$, at the price of a smaller product cache within the 128-qubit limit.

Each circuit plans its products in phases over a cache of low-address products, with the product basis chosen so that each low product is computed only once. The circuits without coded bits need 225 products and 305 Toffolis each, those with one coded bit per word 203 products and 283 Toffolis, and those with two 180 products and 260 Toffolis.

**Re-paired products.** All fifteen circuits are then regenerated with the products paired again. Features of the low address half and the outputs that share the same output fanout are paired into one Toffoli through the identity above, and the extra low-only and high-only terms are corrected by CNOTs from the existing caches. The relations are laid out on two overlapping hyperplanes of low-address variables plus a remainder, so all eleven low monomials stay live in one phase instead of three. The last block of high addresses has only four valid low addresses, so the unused low coefficients are left free while the relations are built and the four valid values are solved afterwards. This removes three or four products in each circuit with at most one coded bit per word (301 or 302 Toffolis instead of 305 without coded bits, 279 or 280 instead of 283 with one) and two products in each circuit with two (258 instead of 260), where the independent features fall from 360 to 356: the relation spaces of most low-monomial blocks are spanned by hyperplanes, and a few blocks need one or two further directions, found by a joint search across blocks. The full cache needs five more qubits than the 128-qubit limit allows; one CNOT per word clears its top bit from the equal bit below it, and the five freed bits hold cached products and are restored at the end. The circuits are therefore exact on target words whose top two bits are equal, which include every word the circuit family streams from zero and so every word the verifier supplies, but not on arbitrary target words. All fifteen are generated by this notebook's compiler from the frozen product plans; the generator, the plans and the search choices are in the dataset folder `open_circuits_v38/source/`.

A pass costs 4,172 Toffolis, and every circuit is checked on all 1,220 addresses with random target words of the kind it supports, forwards and in reverse. One step of the block encoding costs 16,274 Toffolis: 8,344 for the two lookup passes, 5,311 for the inner and 369 for the outer state preparation, and 2,250 for the Givens rotations.

**Approximation.** The factorization spends the accuracy budget almost fully: both errors sit about 1 μHa below their 20 mHa limits, and $E_\text{gap}$ 0.56 μHa above its required minimum.

## 4. Results

All values come from the official verifiers at full shot counts in this run. The decoder's failure count and the mean Toffoli counts depend slightly on the verification seed.

In [ ]:
r = RESULTS
rows = [
    ('Inverse square root', r['invsqrt']['num_qubits'], f"{r['invsqrt']['toffoli_count']:,.0f}",
     f"C = {r['invsqrt']['toffoli_equivalent']:,.0f} ({r['invsqrt']['rotation_count']:.0f} rotations)", 'invsqrt'),
    ('DQI decoder', r['dqi']['num_qubits'], f"{r['dqi']['toffoli_count']:,.0f}",
     f"R = {r['dqi']['repetition_factor']:.3f} ({r['dqi']['value_errors'] + r['dqi']['phase_errors']} failed of {r['dqi']['verified_shots']:,})", 'dqi'),
    ('FeMoCo lookups', r['dfthcsos']['num_qubits'], f"{r['dfthcsos']['toffoli_count']:,.0f}",
     f"λ_eff = {r['dfthcsos']['lambda_eff']:.2f}", 'dfthcsos'),
]
table = pd.DataFrame([{'Task': name, 'Qubits': q, 'Toffoli': t, 'Other': other,
                       'Cost': f"{r[k]['score']:.4g}", 'Reference cost': f'{REFERENCE[k]:.4g}',
                       'Reference / cost': f"{REFERENCE[k] / r[k]['score']:.2f}"} for name, q, t, other, k in rows])
score = math.prod(REFERENCE[k] / r[k]['score'] for k in REFERENCE) ** (1 / 3)
display(table.style.hide(axis='index'))
print(f'Score (geometric mean of reference / cost): {score:.3f}')

## Sources and reproducibility

The notebook runs on a CPU with the competition data and the attached dataset. The inverse-square-root and DQI circuits are unchanged public Apache-2.0 files; the chemistry weights and the fifteen lookup circuits are produced by this notebook's methods described above. All are stored in the dataset folders `open_circuits_v31/`, `open_circuits_v38/` and `open_circuits_v39/` with their licence and credits and checked against SHA-256 values written in the code cell; the decoder file is stored xz-compressed. The code cell writes `submission.zip` only after all three official verifiers pass.

- Organizers: task definitions, verifiers, kickmix and the log-depth Givens tree ([competition](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation), [arXiv:2610.23832](https://arxiv.org/abs/2610.23832)).
- Inverse square root: the 63-qubit circuit by [mrmeows](https://www.kaggle.com/datasets/mrmeows/qcc-1375-open-circuits) with the main lookup and one product window re-synthesized by [busyaprime](https://www.kaggle.com/datasets/busyaprime/33-qcc-v8-members) (Akmal Xodarev), on the public line of busyaprime (78 qubits), Phuc Thai (EXP-222/226a), this notebook's earlier 84-qubit construction and marsyas (fused phase stage), which in turn builds on Sumit Chongder, Rudra, Artem1410, kozykappa, zongaobian, lightningv08 and Mizero.
- DQI: the 889-qubit decoder of the [SomeJoint team](https://www.kaggle.com/code/somejoint/quantum-circuits-22-77-local-gm-889-qubit-dqi) (Agniv Banerjee), with its streamed conversion, on the fused confluent Hermite decoder with GPT-6 Astra Pro's 658-Toffoli jet, the 889-qubit narrowing by heliosli, and the narrowing kernels of [Ahmed Berat Özer](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression)'s version 9, on work by Alex Chilton and jday96314, Ahmed Berat Özer's Bézout conversion and manwithacat's classical-mask projection, with the earlier in-place continued-fraction and paired-extraction line (somejoint, Phuc Thai, ViberTeam, anazemcev and this notebook's round-12 decoder and cost-driven extraction schedule), going back to the reversible decoder of A. Gu et al., [arXiv:2610.22367](https://arxiv.org/abs/2610.22367).
- FeMoCo: the $(8, 143, 60)$ factorization with coded low bits and the fifteen full-XOR angle circuits by the [SomeJoint team](https://www.kaggle.com/code/somejoint/quantum-circuits-22-67-local-gm-full-xor-chem) (Agniv Banerjee); the exact coefficient gauge by [kirilltym](https://www.kaggle.com/code/kirilltym/quantum-circuits-exact-gauge-5-word-qrom), continued here with multi-column rotations; the coefficient fit of [Ahmed Berat Özer](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression) used as the starting point of our last refit, after earlier gauge studies by kozykappa, Phuc Thai, anazemcev, Alex Chilton and [avazaku](https://www.kaggle.com/code/avazaku/google-quantum-ai-r8-block-gauge-5-word-qrom); phased product plans and the re-pairing of products with a single live low cache after [heliosli](https://www.kaggle.com/code/heliosli/the-dataset-is-now-public-now-enjoy), re-implemented here; rank-one feature spaces after mrmeows; on the earlier chemistry line of Michael Rustad, [Ahmed Berat Özer](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression), busyaprime, lightningv08, [mrmeows](https://www.kaggle.com/datasets/mrmeows/qcc-1375-open-circuits), marculera and this notebook's earlier factorizations and lookup compilers.
- C. Gidney, [Halving the cost of quantum addition](https://quantum-journal.org/papers/q-2018-06-18-74/) (2018); G. H. Low et al., [PRX 15, 041016](https://journals.aps.org/prx/abstract/10.1103/pb2g-j9cw) (2025).

All reused circuits and code are Apache 2.0; this notebook and its dataset are released under Apache 2.0.

In [ ]:
P.write_zip(SUB, WORK / 'submission.zip')